#### Steps:
1. Read bronze races table.
2. Keep only columns required for analytics, will drop the url column.
3. Standardize the column name using snake case (ex: circuitID as circuit_id)
4. Renaming columns to make it more meaningful.
5. Remove duplicate records.
6. Transform values of columns to title case (ex: races names).
7. Write the transform data to silver races table.


In [0]:
%run ../environment_config

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.races"
silver_table = f"{catalog}.{silver_schema}.races"

In [0]:
from pyspark.sql import functions as F

In [0]:
races_df = spark.read.table(bronze_table)

In [0]:
races_selected_df = races_df.select("*").drop(F.col("url"))

In [0]:
## STANDARDIZING COLUMN NAMES
races_renamed_df = races_selected_df.withColumnsRenamed({
                     "raceName" : "race_name",
                     "circuitId": "circuit_id",
                     "date": "race_date"                
})

In [0]:
## REMOVING DUPS BASED ON COMPOSITE PRIMARY KEY
races_distinct_df = races_renamed_df.dropDuplicates(["season","round"])

In [0]:
races_final_df = races_distinct_df.withColumns({
                       "race_name": F.initcap(F.col("race_name")),
                       "circuit_id": F.initcap(F.col("circuit_id"))
})

In [0]:
display(races_final_df)

In [0]:
(races_final_df.write
               .format("delta")
               .option("overwriteschema","true")
               .mode("overwrite")
               .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table)) 